# LeetCode 617 — Merge Two Binary Trees bằng Python

**Link đề:** https://leetcode.com/problems/merge-two-binary-trees/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search, Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def mergeTrees(self, root1: TreeNode | None, root2: TreeNode | None) -> TreeNode | None:
        ...
```

## Mục tiêu học
1. Nắm **cây nhị phân**: `TreeNode`, `left`, `right`, cách LeetCode ghi cây theo tầng `[1,3,2,5]`, tự viết hàm dựng/in/chuyển ngược cây.
2. Hiểu quy tắc gộp: hai node **chồng nhau** → cộng giá trị; chỉ một bên có → **dùng luôn** node đó.
3. Viết 4 lời giải: đệ quy **tạo cây mới** → đệ quy **sửa tại chỗ** `root1` → **stack** lặp → **queue (BFS)** lặp.
4. Hiểu khái niệm **chia sẻ node (aliasing)**: cây kết quả có thể dùng chung node với cây đầu vào – hệ quả và khi nào cần tránh.
5. Trace call stack, chứng minh quy nạp, Time/Space theo `min`/`max` số node, 6 bài tập có đáp án cuối.

## 1. Đề bài và ví dụ

Cho hai cây nhị phân `root1`, `root2`. Hãy tưởng tượng đặt cây này **chồng lên** cây kia (khớp từ gốc). Gộp thành cây mới theo luật:
- Hai node **chồng nhau** → node mới có giá trị **tổng**.
- Chỉ **một** bên có node → dùng node **không rỗng** đó.

Trả về cây đã gộp. Việc gộp **bắt đầu từ hai gốc**.

```text
root1:      1            root2:      2              kết quả:     3
           / \                      / \                         / \
          3   2                    1   3                       4   5
         /                          \   \                     / \   \
        5                            4   7                   5   4   7
```

| root1 | root2 | Output |
|---|---|---|
| `[1,3,2,5]` | `[2,1,3,null,4,null,7]` | `[3,4,5,5,4,null,7]` |
| `[1]` | `[1,2]` | `[2,2]` |

**Constraints:** số node mỗi cây trong `[0, 2000]` (có thể **rỗng**); `-10^4 <= Node.val <= 10^4`.

## 2. Nền tảng: `TreeNode`, dựng cây từ list và in cây

### 2.1. Node nhị phân
Mỗi node có `val`, `left`, `right` (con trái/phải là `TreeNode` hoặc `None`).

### 2.2. Cách LeetCode ghi cây
Theo tầng, từ trái sang phải; `null` (= `None`) đánh dấu **vị trí trống** có cha. Các `null` ở **cuối** được bỏ đi. Ví dụ `[2,1,3,null,4,null,7]`:
- `2` là gốc; con của 2: `1`, `3`;
- con của 1: `null` (trái trống), `4`;
- con của 3: `null`, `7`.

Để dựng lại ta dùng **queue** `deque`: node tạo trước được "nhận con" trước.

In [1]:
from collections import deque
import random


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values):
    """List kiểu LeetCode -> cây."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_list(root):
    """Cây -> list kiểu LeetCode (bỏ None ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def show_tree(root, prefix="", label="root"):
    """In cây nằm ngang: mỗi tầng thụt vào; L = con trái, R = con phải."""
    if root is None:
        return
    print(f"{prefix}{label}: {root.val}")
    show_tree(root.left, prefix + "    ", "L")
    show_tree(root.right, prefix + "    ", "R")


t2 = build_tree([2, 1, 3, None, 4, None, 7])
show_tree(t2)
print("to_list:", to_list(t2))

root: 2
    L: 1
        R: 4
    R: 3
        R: 7
to_list: [2, 1, 3, None, 4, None, 7]


### 2.3. Tham chiếu (reference) và chia sẻ node
Biến Python **trỏ tới** đối tượng. `a = node; b = node` → `a` và `b` là **cùng một** node; sửa qua `a` thì `b` thấy ngay. `is` kiểm tra "cùng đối tượng".

In [2]:
x = TreeNode(5)
y = x                 # không copy, chỉ thêm một cái tên
y.val = 99
print("x.val =", x.val, "| x is y:", x is y)

x.val = 99 | x is y: True


## 3. Tính tay trước khi code

Gộp theo **từng vị trí** tương ứng (gốc với gốc, con trái với con trái, …):

| Vị trí | root1 | root2 | kết quả |
|---|---|---|---|
| gốc | 1 | 2 | 1+2 = **3** |
| trái | 3 | 1 | **4** |
| phải | 2 | 3 | **5** |
| trái-trái | 5 | (trống) | dùng **5** của root1 |
| trái-phải | (trống) | 4 | dùng **4** của root2 |
| phải-trái | (trống) | (trống) | trống |
| phải-phải | (trống) | 7 | dùng **7** |

Nhận xét: khi một bên trống, ta lấy **nguyên cả cây con** của bên kia (không cần đi sâu hơn). Đây là chìa khóa để lời giải ngắn.

## 4. Cách 1 — Đệ quy tạo cây **mới** (không sửa input)

### 4.1. Đệ quy trên cây
Bài toán trên cây có cấu trúc tự lặp: gộp hai cây = gộp gốc + gộp hai cây con trái + gộp hai cây con phải.

- **Base case:** một trong hai rỗng → trả cây kia (có thể cũng rỗng).
- **Bước đệ quy:** tạo node mới `TreeNode(a.val + b.val)`, `left = merge(a.left, b.left)`, `right = merge(a.right, b.right)`.

Ở base case, nếu muốn **hoàn toàn** không chia sẻ node với input, ta copy cây con (`clone`). Bản dưới đây copy để minh họa rõ "cây mới".

In [3]:
def clone(node):
    if node is None:
        return None
    return TreeNode(node.val, clone(node.left), clone(node.right))


def merge_new(a, b):
    if a is None:
        return clone(b)
    if b is None:
        return clone(a)
    node = TreeNode(a.val + b.val)
    node.left = merge_new(a.left, b.left)
    node.right = merge_new(a.right, b.right)
    return node


r1, r2 = build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7])
merged = merge_new(r1, r2)
print(to_list(merged))                    # [3, 4, 5, 5, 4, None, 7]
print("root1 không đổi:", to_list(r1))
print(to_list(merge_new(build_tree([1]), build_tree([1, 2]))))   # [2, 2]

[3, 4, 5, 5, 4, None, 7]
root1 không đổi: [1, 3, 2, 5]
[2, 2]


### 4.2. TRACE call stack
In thụt lề theo độ sâu: thấy rõ khi nào gặp base case và trả nguyên cây con.

In [4]:
def trace_merge(a, b, depth=0):
    pad = "  " * depth
    av = a.val if a else None
    bv = b.val if b else None
    if a is None or b is None:
        res = a if a else b
        print(f"{pad}merge({av}, {bv}) -> một bên rỗng, trả cây con {to_list(res)}")
        return res
    print(f"{pad}merge({av}, {bv}) -> tạo node {av + bv}")
    node = TreeNode(a.val + b.val)
    node.left = trace_merge(a.left, b.left, depth + 1)
    node.right = trace_merge(a.right, b.right, depth + 1)
    return node

res = trace_merge(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))
print("Kết quả:", to_list(res))

merge(1, 2) -> tạo node 3
  merge(3, 1) -> tạo node 4
    merge(5, None) -> một bên rỗng, trả cây con [5]
    merge(None, 4) -> một bên rỗng, trả cây con [4]
  merge(2, 3) -> tạo node 5
    merge(None, None) -> một bên rỗng, trả cây con []
    merge(None, 7) -> một bên rỗng, trả cây con [7]
Kết quả: [3, 4, 5, 5, 4, None, 7]


## 5. Cách 2 — Đệ quy **sửa tại chỗ** `root1` (lời giải kinh điển)

Thay vì tạo node mới, cộng thẳng vào `a`: `a.val += b.val`, rồi `a.left = merge(a.left, b.left)`, `a.right = merge(a.right, b.right)`, trả `a`. Base case trả **nguyên** cây con không rỗng (không copy).

Ít bộ nhớ hơn, code ngắn nhất – nhưng **thay đổi `root1`** và cây kết quả **dùng chung node** với `root2` ở những nhánh chỉ `root2` có.

In [5]:
def merge_inplace(a, b):
    if a is None:
        return b
    if b is None:
        return a
    a.val += b.val
    a.left = merge_inplace(a.left, b.left)
    a.right = merge_inplace(a.right, b.right)
    return a


r1, r2 = build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7])
merged = merge_inplace(r1, r2)
print("merged          :", to_list(merged))
print("merged is r1    :", merged is r1, "-> root1 đã bị sửa:", to_list(r1))
print("nút 7 dùng chung:", merged.right.right is r2.right.right)

merged          : [3, 4, 5, 5, 4, None, 7]
merged is r1    : True -> root1 đã bị sửa: [3, 4, 5, 5, 4, None, 7]
nút 7 dùng chung: True


### 5.1. Hệ quả của việc chia sẻ node
Nếu sau đó ta sửa cây kết quả, `root2` cũng bị đổi theo ở phần dùng chung. Trên LeetCode điều này **không sao** (họ chỉ đọc kết quả), nhưng trong dự án thật cần cân nhắc.

In [6]:
merged.right.right.val = 1000
print("Sửa merged xong, root2 thành:", to_list(r2))

Sửa merged xong, root2 thành: [2, 1, 3, None, 4, None, 1000]


## 6. Nút thắt của đệ quy và hướng lặp

Đệ quy dùng call stack sâu bằng **chiều cao** cây. Với 2000 node, cây lệch (dạng chuỗi) cao 2000 > giới hạn mặc định ~1000 của Python → có thể `RecursionError` khi chạy ở máy cá nhân. Cách lặp tự quản lý stack/queue để tránh điều này.

In [7]:
import sys

def chain(n):
    """Cây lệch trái n node."""
    root = cur = TreeNode(1)
    for _ in range(n - 1):
        cur.left = TreeNode(1)
        cur = cur.left
    return root

print("Giới hạn đệ quy:", sys.getrecursionlimit())
try:
    merge_inplace(chain(3000), chain(3000))
    print("Đệ quy chạy được")
except RecursionError:
    print("RecursionError với cây lệch 3000 node")

Giới hạn đệ quy: 1000
RecursionError với cây lệch 3000 node


## 7. Cách 3 — Lặp bằng **stack** các cặp node (sửa tại chỗ)

Ý tưởng: stack chứa các **cặp** `(n1, n2)` đều **khác `None`**, nghĩa là "hai node chồng nhau cần cộng".

Giảng từng đoạn:
1. Xử lý rỗng: `if not root1: return root2`.
2. `stack = [(root1, root2)]`.
3. Pop `(n1, n2)`: `n1.val += n2.val`.
4. Con trái: nếu `n1.left` và `n2.left` đều có → đẩy cặp. Nếu `n1.left` rỗng → **gắn** `n1.left = n2.left` (lấy nguyên cây con). Nếu chỉ `n2.left` rỗng → giữ nguyên.
5. Con phải tương tự.

In [8]:
def merge_stack(root1, root2):
    if root1 is None:
        return root2
    if root2 is None:
        return root1
    stack = [(root1, root2)]
    while stack:
        n1, n2 = stack.pop()
        n1.val += n2.val
        # con trái
        if n1.left and n2.left:
            stack.append((n1.left, n2.left))
        elif n1.left is None:
            n1.left = n2.left
        # con phải
        if n1.right and n2.right:
            stack.append((n1.right, n2.right))
        elif n1.right is None:
            n1.right = n2.right
    return root1


print(to_list(merge_stack(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))))
print(to_list(merge_stack(build_tree([1]), build_tree([1, 2]))))
print("Cây lệch 3000 node:", to_list(merge_stack(chain(3000), chain(3000)))[:3], "...")

[3, 4, 5, 5, 4, None, 7]
[2, 2]
Cây lệch 3000 node: [2, 2, None] ...


### 7.1. TRACE stack
| Bước | pop `(n1,n2)` | n1.val mới | xử lý trái | xử lý phải | stack sau |
|---:|---|---:|---|---|---|
| 1 | (1, 2) | 3 | cả hai có → đẩy (3,1) | cả hai có → đẩy (2,3) | `[(3,1),(2,3)]` |
| 2 | (2, 3) | 5 | cả hai rỗng → giữ | n1 rỗng → gắn 7 | `[(3,1)]` |
| 3 | (3, 1) | 4 | n2 rỗng → giữ 5 | n1 rỗng → gắn 4 | `[]` |

In [9]:
def trace_merge_stack(root1, root2):
    stack = [(root1, root2)]
    step = 1
    while stack:
        n1, n2 = stack.pop()
        n1.val += n2.val
        acts = []
        for side in ("left", "right"):
            c1, c2 = getattr(n1, side), getattr(n2, side)
            if c1 and c2:
                stack.append((c1, c2))
                acts.append(f"{side}: đẩy ({c1.val},{c2.val})")
            elif c1 is None and c2 is not None:
                setattr(n1, side, c2)
                acts.append(f"{side}: gắn cây con {to_list(c2)}")
            else:
                acts.append(f"{side}: giữ nguyên")
        print(f"Bước {step}: n1.val={n1.val} | " + " ; ".join(acts) +
              f" | stack={[(p.val, q.val) for p, q in stack]}")
        step += 1
    return root1

print("Kết quả:", to_list(trace_merge_stack(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))))

Bước 1: n1.val=3 | left: đẩy (3,1) ; right: đẩy (2,3) | stack=[(3, 1), (2, 3)]
Bước 2: n1.val=5 | left: giữ nguyên ; right: gắn cây con [7] | stack=[(3, 1)]
Bước 3: n1.val=4 | left: giữ nguyên ; right: gắn cây con [4] | stack=[]
Kết quả: [3, 4, 5, 5, 4, None, 7]


`getattr(obj, "left")` đọc thuộc tính theo tên chuỗi, `setattr(obj, "left", v)` gán – giúp viết một vòng cho cả hai phía trái/phải. (Lưu ý: giá trị cặp in trong stack là giá trị **trước** khi cộng.)

## 8. Cách 4 — Lặp bằng **queue** (BFS, theo tầng)

Giống hệt cách 3 nhưng thay stack bằng `deque` + `popleft`: các cặp được xử lý **theo tầng** (gốc → tầng 1 → tầng 2…). Kết quả cuối giống nhau vì mỗi cặp độc lập với nhau.

In [10]:
def merge_bfs(root1, root2):
    if root1 is None:
        return root2
    if root2 is None:
        return root1
    queue = deque([(root1, root2)])
    while queue:
        n1, n2 = queue.popleft()
        n1.val += n2.val
        if n1.left and n2.left:
            queue.append((n1.left, n2.left))
        elif n1.left is None:
            n1.left = n2.left
        if n1.right and n2.right:
            queue.append((n1.right, n2.right))
        elif n1.right is None:
            n1.right = n2.right
    return root1

print(to_list(merge_bfs(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))))

[3, 4, 5, 5, 4, None, 7]


## 9. Code nộp LeetCode

Bản chính: đệ quy sửa tại chỗ (ngắn, chuẩn phỏng vấn). LeetCode đã có `TreeNode`.

In [11]:
class Solution:
    def mergeTrees(self, root1: TreeNode | None, root2: TreeNode | None) -> TreeNode | None:
        if root1 is None:
            return root2
        if root2 is None:
            return root1
        root1.val += root2.val
        root1.left = self.mergeTrees(root1.left, root2.left)
        root1.right = self.mergeTrees(root1.right, root2.right)
        return root1


sol = Solution()
assert to_list(sol.mergeTrees(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))) == [3, 4, 5, 5, 4, None, 7]
assert to_list(sol.mergeTrees(build_tree([1]), build_tree([1, 2]))) == [2, 2]
assert sol.mergeTrees(None, None) is None
print("Solution PASS ví dụ đề")

Solution PASS ví dụ đề


Bản lặp (an toàn với cây rất sâu):

In [12]:
class SolutionIterative:
    def mergeTrees(self, root1: TreeNode | None, root2: TreeNode | None) -> TreeNode | None:
        if not root1 or not root2:
            return root1 or root2
        stack = [(root1, root2)]
        while stack:
            n1, n2 = stack.pop()
            n1.val += n2.val
            if n1.left and n2.left:
                stack.append((n1.left, n2.left))
            elif not n1.left:
                n1.left = n2.left
            if n1.right and n2.right:
                stack.append((n1.right, n2.right))
            elif not n1.right:
                n1.right = n2.right
        return root1

assert to_list(SolutionIterative().mergeTrees(build_tree([1]), build_tree([1, 2]))) == [2, 2]
print("SolutionIterative PASS")

SolutionIterative PASS


`root1 or root2`: `or` trả về **toán hạng** đầu tiên "truthy" (không phải `True/False`). Node là truthy, `None` là falsy → trả node không rỗng, hoặc `None` nếu cả hai rỗng.

## 10. Vì sao đúng? (quy nạp)

Gọi `M(a, b)` là cây gộp đúng theo định nghĩa. Chứng minh `merge(a, b) = M(a, b)` bằng **quy nạp theo tổng số node** của `a` và `b`:
- **Cơ sở:** `a` hoặc `b` rỗng → theo luật, mọi vị trí chỉ có một bên → `M` chính là cây không rỗng. Code trả đúng cây đó.
- **Bước:** cả hai khác rỗng → gốc chồng nhau → giá trị `a.val + b.val`. Cây con trái của `M` là gộp hai cây con trái (luật áp dụng độc lập cho từng vị trí); theo giả thiết quy nạp (ít node hơn), `merge(a.left, b.left)` đúng. Tương tự bên phải.

**Cách lặp – invariant:** mọi cặp trong stack là hai node khác rỗng **ở cùng vị trí** chưa được cộng; mọi vị trí đã xử lý xong trong cây `root1` đều đã mang giá trị/cây con đúng của `M`. Khi stack rỗng, mọi vị trí chồng nhau đã được cộng, vị trí chỉ có `root2` đã được gắn → `root1 = M`.

## 11. Time / Space Complexity và so sánh

`m`, `n` = số node hai cây; `h` = chiều cao (của phần chồng nhau).

| Cách | Time | Space phụ | Sửa input? | Chia sẻ node? |
|---|---|---|---|---|
| 1. Đệ quy tạo mới + clone | `O(m + n)` | `O(m + n)` cho cây mới, `O(h)` stack | Không | Không |
| 2. Đệ quy tại chỗ | `O(min(m, n))` | `O(h)` call stack | Có (`root1`) | Có |
| 3. Stack lặp | `O(min(m, n))` | `O(min(m, n))` tệ nhất | Có | Có |
| 4. Queue BFS | `O(min(m, n))` | `O(độ rộng)` | Có | Có |

Vì sao cách 2–4 chỉ `O(min(m, n))`: ta chỉ đi vào những vị trí **cả hai** cây đều có node; phần chỉ một bên có được gắn nguyên trong `O(1)`.

## 12. Bộ test: mẫu đề, biên, đối chiếu ngẫu nhiên

Lưu ý: cách 2–4 **sửa** `root1`, nên mỗi lần chạy phải dựng cây mới từ list.

In [13]:
methods = {
    "new": merge_new,
    "inplace": merge_inplace,
    "stack": merge_stack,
    "bfs": merge_bfs,
    "Solution": Solution().mergeTrees,
    "SolutionIterative": SolutionIterative().mergeTrees,
}
cases = [
    ("đề 1", [1, 3, 2, 5], [2, 1, 3, None, 4, None, 7], [3, 4, 5, 5, 4, None, 7]),
    ("đề 2", [1], [1, 2], [2, 2]),
    ("cả hai rỗng", [], [], []),
    ("root1 rỗng", [], [1, None, 2], [1, None, 2]),
    ("root2 rỗng", [4, 5], [], [4, 5]),
    ("số âm triệt tiêu", [5, -3], [-5, 3], [0, 0]),
    ("lệch hai phía", [1, 2, None, 3], [1, None, 2, None, 3], [2, 2, 2, 3, None, None, 3]),
]
for name, l1, l2, expected in cases:
    for mname, f in methods.items():
        got = to_list(f(build_tree(l1), build_tree(l2)))
        assert got == expected, (name, mname, got)
    print(f"{name:<18} PASS -> {expected}")

đề 1               PASS -> [3, 4, 5, 5, 4, None, 7]
đề 2               PASS -> [2, 2]
cả hai rỗng        PASS -> []
root1 rỗng         PASS -> [1, None, 2]
root2 rỗng         PASS -> [4, 5]
số âm triệt tiêu   PASS -> [0, 0]
lệch hai phía      PASS -> [2, 2, 2, 3, None, None, 3]


In [14]:
def random_tree(n, rng):
    if n == 0:
        return None
    root = TreeNode(rng.randint(-9, 9))
    nodes = [root]
    while len(nodes) < n:
        parent = rng.choice(nodes)
        side = rng.choice(["left", "right"])
        if getattr(parent, side) is None:
            child = TreeNode(rng.randint(-9, 9))
            setattr(parent, side, child)
            nodes.append(child)
    return root

def merge_by_positions(l1, l2):
    """Oracle độc lập: đánh số vị trí kiểu heap (gốc 1, trái 2i, phải 2i+1) rồi cộng."""
    def positions(root):
        pos, stack = {}, [(root, 1)]
        while stack:
            node, k = stack.pop()
            if node:
                pos[k] = node.val
                stack += [(node.left, 2 * k), (node.right, 2 * k + 1)]
        return pos
    p1, p2 = positions(build_tree(l1)), positions(build_tree(l2))
    return {k: p1.get(k, 0) + p2.get(k, 0) for k in p1.keys() | p2.keys()}

rng = random.Random(617)
for _ in range(300):
    l1 = to_list(random_tree(rng.randint(0, 12), rng))
    l2 = to_list(random_tree(rng.randint(0, 12), rng))
    expected = merge_by_positions(l1, l2)
    outs = []
    for mname, f in methods.items():
        res = f(build_tree(l1), build_tree(l2))
        outs.append(to_list(res))
        assert merge_by_positions(to_list(res), []) == expected, (l1, l2, mname)
    assert all(o == outs[0] for o in outs)
print("300 cặp cây ngẫu nhiên: 6 cách khớp oracle đánh số vị trí — PASS")

300 cặp cây ngẫu nhiên: 6 cách khớp oracle đánh số vị trí — PASS


`p1.keys() | p2.keys()` là **hợp** hai tập key (dict keys hỗ trợ phép toán tập hợp).

## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra `root1 is None`
Quên `root2 is None` → truy cập `root2.val` lỗi `AttributeError`.

### Lỗi 2: Cộng nhưng **không gán lại** con
Viết `merge(a.left, b.left)` mà không `a.left = ...` → khi `a.left` rỗng, cây con của `b` không được gắn vào.

### Lỗi 3: Khi một bên rỗng, tạo node mới với giá trị 0 + val rồi đi tiếp
Không sai kết quả nhưng tốn công vô ích; và nếu viết `TreeNode(0)` thay cho `None` sẽ **thêm node thừa**.

In [15]:
def merge_wrong_no_assign(a, b):
    if a is None:
        return b
    if b is None:
        return a
    a.val += b.val
    merge_wrong_no_assign(a.left, b.left)       # quên a.left = ...
    merge_wrong_no_assign(a.right, b.right)
    return a

def merge_wrong_dummy(a, b):
    if a is None and b is None:
        return None
    a = a or TreeNode(0)                        # tạo node giả
    b = b or TreeNode(0)
    node = TreeNode(a.val + b.val)
    node.left = merge_wrong_dummy(a.left, b.left)
    node.right = merge_wrong_dummy(a.right, b.right)
    return node

print("Quên gán con:", to_list(merge_wrong_no_assign(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))))
print("Node giả 0   :", to_list(merge_wrong_dummy(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))),
      "<- vẫn đúng ở đây nhưng duyệt hết m+n node")
try:
    def merge_wrong_one_check(a, b):
        if a is None:
            return b
        a.val += b.val
        return a
    merge_wrong_one_check(build_tree([1]), None)
except AttributeError as e:
    print("Chỉ kiểm tra một bên:", type(e).__name__)

Quên gán con: [3, 4, 5, 5]
Node giả 0   : [3, 4, 5, 5, 4, None, 7] <- vẫn đúng ở đây nhưng duyệt hết m+n node
Chỉ kiểm tra một bên: AttributeError


### Lỗi 4: Tưởng `root1` không bị đổi
Sau khi gọi bản tại chỗ, `root1` **đã** là cây gộp. Nếu cần giữ input, dùng cách 1.

### Lỗi 5: So sánh cây bằng `==`
Hai `TreeNode` khác đối tượng nhưng giống nội dung vẫn `!=` (mặc định `==` so sánh danh tính). Muốn so sánh nội dung hãy chuyển sang list bằng `to_list` (hoặc viết hàm `same_tree`).

In [16]:
print("TreeNode(1) == TreeNode(1):", TreeNode(1) == TreeNode(1))
print("So bằng to_list           :", to_list(TreeNode(1)) == to_list(TreeNode(1)))

TreeNode(1) == TreeNode(1): False
So bằng to_list           : True


## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu "duyệt song song hai cây":** đề đưa **hai** cây và so khớp **từng vị trí** (gộp, so sánh, đối xứng). Khung chung:

```python
def f(a, b):
    if a is None or b is None:
        return xử_lý_biên(a, b)
    ... a.val, b.val ...
    f(a.left, b.left); f(a.right, b.right)
```

**Bài cùng khung:** 100. Same Tree, 101. Symmetric Tree (so `a.left` với `b.right`), 572. Subtree of Another Tree, 951. Flip Equivalent Binary Trees.

**Biến thể:** gộp bằng `max` thay vì tổng; gộp `k` cây; chỉ giữ vị trí **cả hai** đều có (giao).

## 15. Bài tập (TODO – không gọi khi chưa làm)

**Bài 1 — Tính tay.** Gộp `[1,2,3]` và `[4,null,5,null,6]`, viết kết quả dạng list LeetCode.

**Bài 2 — Đoán output.** Sau `m = merge_inplace(r1, r2)` với `r1 = build_tree([1])`, `r2 = build_tree([1, 2])`: `m is r1`? `m.left is r2.left`?

**Bài 3 — Viết lại** `mergeTrees` đệ quy tại chỗ.

**Bài 4 — Gộp bằng max:** hai node chồng nhau lấy `max` thay vì tổng; tạo cây mới.

**Bài 5 — Gộp `k` cây:** `merge_k(trees)` (list có thể chứa `None`).

**Bài 6 — Mini challenge (giao):** chỉ giữ những vị trí **cả hai** cây đều có node; giá trị là tổng. Tạo cây mới.

In [17]:
answer_ex1 = None      # TODO: list LeetCode
answer_ex2 = None      # TODO: (True/False, True/False)


def merge_exercise(a, b):
    # TODO 1: một bên rỗng -> trả bên kia
    # TODO 2: cộng giá trị vào a
    # TODO 3: GÁN a.left, a.right bằng lời gọi đệ quy
    pass


def merge_max(a, b):
    # TODO: tạo node mới với max(a.val, b.val) khi cả hai có
    pass

# Kiểm tra sau khi làm:
# assert to_list(merge_exercise(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))) == [3, 4, 5, 5, 4, None, 7]
# assert to_list(merge_max(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))) == [2, 3, 3, 5, 4, None, 7]

In [18]:
def merge_k(trees):
    # TODO: gộp dần từng cây (dùng một hàm gộp 2 cây)
    pass


def merge_intersection(a, b):
    # TODO: một bên rỗng -> None
    pass

# Kiểm tra sau khi làm:
# assert to_list(merge_k([build_tree([1]), None, build_tree([1, 2]), build_tree([1, None, 3])])) == [3, 2, 3]
# assert to_list(merge_intersection(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))) == [3, 4, 5]

## 16. Gợi ý 3 mức

**Bài 1:** (1) Gộp từng vị trí. (2) Gốc 1+4; trái chỉ có `2`; phải 3+5; con phải của phải chỉ có `6`. (3) Viết theo tầng, nhớ `null` cho vị trí trống có cha.

**Bài 2:** (1) Bản tại chỗ trả `root1`. (2) `r1.left` rỗng nên được gắn `r2.left` nguyên vẹn. (3) Cả hai đều `True`.

**Bài 3:** (1) Base case hai dòng. (2) `a.val += b.val`. (3) `a.left = merge_exercise(a.left, b.left)`.

**Bài 4:** (1) Giống cách 1 (tạo mới). (2) `TreeNode(max(a.val, b.val))`. (3) Base case trả `clone` bên không rỗng.

**Bài 5:** (1) Gộp lần lượt: `res = merge(res, t)`. (2) Khởi tạo `res = None`. (3) Dùng bản tạo mới để không sửa các cây đầu vào.

**Bài 6:** (1) Base case: **một bên rỗng → `None`** (khác bài gốc). (2) Tạo node tổng. (3) Đệ quy hai phía.

## 17. Đáp án tham khảo (chạy kiểm tra)

In [19]:
ans_ex1 = to_list(merge_new(build_tree([1, 2, 3]), build_tree([4, None, 5, None, 6])))   # [5, 2, 8, None, None, None, 6]
r1, r2 = build_tree([1]), build_tree([1, 2])
m = merge_inplace(r1, r2)
ans_ex2 = (m is r1, m.left is r2.left)                                                   # (True, True)


def ans_merge_exercise(a, b):
    if a is None:
        return b
    if b is None:
        return a
    a.val += b.val
    a.left = ans_merge_exercise(a.left, b.left)
    a.right = ans_merge_exercise(a.right, b.right)
    return a


def ans_merge_max(a, b):
    if a is None:
        return clone(b)
    if b is None:
        return clone(a)
    return TreeNode(max(a.val, b.val), ans_merge_max(a.left, b.left), ans_merge_max(a.right, b.right))


def ans_merge_k(trees):
    res = None
    for t in trees:
        res = merge_new(res, t)
    return res


def ans_merge_intersection(a, b):
    if a is None or b is None:
        return None
    return TreeNode(a.val + b.val,
                    ans_merge_intersection(a.left, b.left),
                    ans_merge_intersection(a.right, b.right))


print("Bài 1:", ans_ex1, "| Bài 2:", ans_ex2)
assert ans_ex1 == [5, 2, 8, None, None, None, 6]
assert ans_ex2 == (True, True)
assert to_list(ans_merge_exercise(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))) == [3, 4, 5, 5, 4, None, 7]
assert to_list(ans_merge_max(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))) == [2, 3, 3, 5, 4, None, 7]
assert to_list(ans_merge_k([build_tree([1]), None, build_tree([1, 2]), build_tree([1, None, 3])])) == [3, 2, 3]
assert ans_merge_k([]) is None
assert to_list(ans_merge_intersection(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))) == [3, 4, 5]
print("Đáp án: tất cả test PASS")

Bài 1: [5, 2, 8, None, None, None, 6] | Bài 2: (True, True)
Đáp án: tất cả test PASS


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def mergeTrees(self, root1, root2):
        if root1 is None:
            return root2                  # một bên rỗng -> lấy nguyên cây kia
        if root2 is None:
            return root1
        root1.val += root2.val            # chồng nhau -> cộng
        root1.left = self.mergeTrees(root1.left, root2.left)     # NHỚ gán lại
        root1.right = self.mergeTrees(root1.right, root2.right)
        return root1
```

Câu nhớ: **"rỗng thì lấy bên kia – chồng thì cộng – gán lại hai con"**.

### Checklist
- [ ] Xử lý **cả hai** trường hợp một bên rỗng (và cả hai rỗng)?
- [ ] Gán lại `root1.left`, `root1.right` bằng kết quả đệ quy?
- [ ] Hiểu rằng bản tại chỗ sửa `root1` và dùng chung node với `root2`?
- [ ] Cây rất sâu → cân nhắc bản lặp bằng stack?
- [ ] Thử cây rỗng, một bên rỗng, giá trị âm, cây lệch?